# 03 - Vector Retrieval from Real Documents

## Objective

In this notebook, we will connect the components built in the
previous notebooks.

Pipeline:

PDF
↓
Text Extraction
↓
Chunking
↓
Embedding
↓
FAISS Index
↓
Query Embedding
↓
Top-K Retrieval

We will also preserve metadata such as:

- source document
- page number
- chunk ID

This metadata will later allow us to provide citations in our RAG
answers.

In [1]:
# 3. Import our own modules
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.loader import load_pdf
from src.chunker import create_page_chunks

In [2]:
# 4. Check our project structure
print("Project root:", PROJECT_ROOT)

data_dir = PROJECT_ROOT / "data"

print("\nFiles in data/:")
for file in data_dir.iterdir():
    print(file.name)

Project root: c:\Users\sansk\OneDrive\Documents\clone\ResearchRAG

Files in data/:
.gitkeep
attention-is-all-you-need-Paper.pdf
chain-of-thought.pdf
deep residual learning for image recognition.pdf
GANs.pdf
Lora.pdf


In [3]:
# 5. load multiple pdfs
pdf_files = list(data_dir.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)


Number of PDFs: 5
- attention-is-all-you-need-Paper.pdf
- chain-of-thought.pdf
- deep residual learning for image recognition.pdf
- GANs.pdf
- Lora.pdf


In [4]:
# 6. load all pdfs
all_pages = []

for pdf_path in pdf_files:
    pages = load_pdf(pdf_path)

    for page in pages:
        page["source"] = pdf_path.name

    all_pages.extend(pages)

print("Total pages:", len(all_pages))

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/KGMNOB+TT93o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(64, 0, 3046514169232), '/LastChar': 121, '/Widths': [333, 0, 0, 507, 0, 507, 0, 0, 507, 507, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 440, 0, 0, 0, 440, 0, 0, 0, 0, 0, 0, 280, 0, 0, 0, 0, 0, 333, 0, 0, 0, 0, 0, 0, 480]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/MKDMEG+TT55o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(312, 0, 3046514169232), '/LastChar': 121, '/Widths': [340, 0, 0, 0, 500, 0, 500, 500, 0, 0, 0, 500, 0, 0, 0, 0, 

Total pages: 72


In [5]:
# 7. Inspect the data
all_pages[0]

print("Source:", all_pages[0]["source"])
print("Page:", all_pages[0]["page"])
print("Text:", all_pages[0]["text"][:500])

Source: attention-is-all-you-need-Paper.pdf
Page: 1
Text: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or


In [7]:
chunks = create_page_chunks(
    all_pages,
    chunk_size=1500,
    overlap_paragraphs=1
)

print("Number of chunks:", len(chunks))

Number of chunks: 76


In [8]:
for i, chunk in enumerate(chunks[:10]):

    print("=" * 100)

    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Characters:", len(chunk["text"]))

    print()
    print(chunk["text"][:1500])

Chunk ID: 0
Source: attention-is-all-you-need-Paper.pdf
Page: 1
Characters: 2908

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while be

In [9]:
print("Total chunks:", len(chunks))

Total chunks: 76


In [10]:
chunks[0]

{'source': 'attention-is-all-you-need-Paper.pdf',
 'page': 1,
 'text': 'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superio

In [ ]:
# 10. Why metadata matters

# This is a subtle but very important RAG design decision.

# We don't want our vector database to contain only:

# vector

# We want:

# vector
#     +
# chunk
#     +
# metadata

# Conceptually:

# ┌──────────────────────────────┐
# │ Vector                       │
# │                              │
# │ [0.21, -0.14, ...]           │
# │                              │
# │ Metadata                     │
# │ source = paper1.pdf         │
# │ page = 7                     │
# │ chunk_id = 31                │
# │                              │
# │ Text                         │
# │ "The proposed method..."     │
# └──────────────────────────────┘

# FAISS itself stores the vectors, so we maintain the metadata separately.

# This is a common pattern in vector retrieval systems.

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
# rebuild embeddings
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
# rebuild faiss
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors indexed:", index.ntotal)

Embedding dimension: 384
Vectors indexed: 76


In [17]:
query = "What is self-attention?"

results = retrieve(
    query,
    model,
    index,
    chunks,
    k=10
)

for rank, result in enumerate(results, start=1):

    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print()
    print(result["text"][:1000])

NameError: name 'retrieve' is not defined

In [ ]:
for rank, result in enumerate(results, start=1):

    print("=" * 100)

    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print()

    print(result["text"][:1000])

Rank: 1
Score: 0.4726
Source: attention-is-all-you-need-Paper.pdf
Page: 2

he number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
textual entailment and learning task-independent sentence representations [4, 22, 23, 19].
End-to-en